## **🧠 Clasificador de SPAM en Español con Deep Learning (TensorFlow/Keras)**

**Introducción:**

Este proyecto tiene como objetivo desarrollar e implementar un **sistema de clasificación de mensajes (SMS/texto) en español** utilizando técnicas de _Deep Learning_. La meta principal es distinguir con alta precisión entre mensajes legítimos (**HAM**) y mensajes no deseados (**SPAM**). Para lograr esto, se emplea una arquitectura de red neuronal recurrente (aunque la implementación usa `GlobalAveragePooling1D` para eficiencia en el clasificador de texto) construida con **TensorFlow y Keras**. El _pipeline_ incluye la **vectorización de texto** mediante la capa `TextVectorization` para construir un vocabulario adaptado al idioma, la creación de un _embedding_ vectorial de palabras, y una capa densa final con activación sigmoide para la clasificación binaria. Este enfoque permite al modelo aprender representaciones semánticas de las palabras clave en español, cruciales para la detección efectiva de SPAM.

He organizado los pasos de la siguiente manera:

1.  **Configuración Inicial:** Importar librerías.
    
2.  **Carga y Preparación de Datos:** Cargar los DataFrames y separar las variables.
    
3.  **Preprocesamiento del Texto:** Definir, adaptar y aplicar la capa `TextVectorization`.
    
4.  **Definición y Compilación del Modelo:** Crear la arquitectura de la red neuronal.
    
5.  **Entrenamiento del Modelo:** Entrenar el modelo con los datos vectorizados.
    
6.  **Prueba del Modelo:** Crear la función de predicción y probar con ejemplos.

## **🧠 PARTE 1: Modelo de SPAM en español (Refactorizado)**

Objetivo: Entrenar un clasificador de SPAM en español y probarlo.

### **1️⃣ Configuración Inicial**

In [1]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras import models
from sklearn.preprocessing import LabelEncoder # Necesario para codificar etiquetas 'spam'/'ham'

In [2]:
print(tf.__version__)

2.19.0


Esta celda se encarga de importar todas las librerías necesarias para el proyecto. Aquí te explico cada una:

-   `import pandas as pd`: Importa la librería  `pandas`, fundamental para la manipulación y análisis de datos, especialmente para trabajar con DataFrames (tablas de datos).
-   `import tensorflow as tf`: Importa  `tensorflow`, la plataforma de código abierto para aprendizaje automático. Es el  _framework_  principal que usaremos para construir el modelo de red neuronal.
-   `from tensorflow.keras.layers import TextVectorization, Embedding, GlobalAveragePooling1D, Dense, Dropout`: Importa capas específicas de Keras (una API de alto nivel de TensorFlow) que son componentes clave de nuestra red neuronal:
    -   `TextVectorization`: Para convertir texto crudo en secuencias numéricas que el modelo pueda entender.
    -   `Embedding`: Para crear representaciones vectoriales densas (embeddings) de las palabras.
    -   `GlobalAveragePooling1D`: Para reducir la dimensionalidad de los embeddings.
    -   `Dense`: Las capas neuronales estándar (totalmente conectadas).
    -   `Dropout`: Una técnica de regularización para evitar el sobreajuste del modelo.
-   `from tensorflow.keras import models`: Importa la funcionalidad  `models`  de Keras, que usaremos para definir la estructura secuencial de nuestra red neuronal.
-   `from sklearn.preprocessing import LabelEncoder`: Importa  `LabelEncoder`  de la librería  `scikit-learn`, que es útil para convertir etiquetas de texto como 'spam' y 'ham' en valores numéricos (0 y 1), un formato que el modelo puede procesar.


### **2️⃣ Carga y Preparación de Datos**

Cargamos los datasets y codificamos las etiquetas de texto (`spam`/`ham`) a valores numéricos (`1`/`0`), que es lo que espera el modelo para una clasificación binaria.

In [4]:
# Cargar datasets (asumiendo que los archivos están disponibles en el entorno)
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# Separar características (X) y etiquetas (y)
X_train = train_df['text'].fillna('') # Rellenar NaN con cadena vacía
y_train_text = train_df['label']

X_test = test_df['text'].fillna('')
y_test_text = test_df['label']

# Codificar las etiquetas 'spam'/'ham' a 1/0
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)
y_test = label_encoder.transform(y_test_text)

print(f"Dataset de Entrenamiento: {len(X_train)} muestras")
print(f"Dataset de Evaluación: {len(X_test)} muestras")
print(f"Etiquetas originales: {label_encoder.classes_}")
print(f"Codificación: {label_encoder.transform(label_encoder.classes_)}")

Dataset de Entrenamiento: 4457 muestras
Dataset de Evaluación: 1115 muestras
Etiquetas originales: [0 1]
Codificación: [0 1]


Esta celda se encarga de cargar los conjuntos de datos, separarlos en características (el texto de los mensajes) y etiquetas (si es spam o no), y de preprocesar estas etiquetas para que el modelo pueda entenderlas. Aquí te detallo cada paso:

1.  **Cargar Datasets (`train_df`  y  `test_df`):**
    
    -   `train_df = pd.read_csv("train.csv")`  y  `test_df = pd.read_csv("test.csv")`  leen los archivos  `train.csv`  y  `test.csv`  (que contienen los datos de entrenamiento y prueba, respectivamente) y los cargan en DataFrames de pandas. Se asume que estos archivos ya están disponibles en el entorno.
2.  **Separar Características (X) y Etiquetas (y):**
    
    -   `X_train = train_df['text'].fillna('')`: Toma la columna 'text' del DataFrame de entrenamiento (`train_df`) como las características (`X_train`). El  `.fillna('')`  es importante para reemplazar cualquier valor nulo (`NaN`) en la columna de texto con una cadena vacía, asegurando que todos los textos sean strings.
    -   `y_train_text = train_df['label']`: Toma la columna 'label' de  `train_df`  como las etiquetas (`y_train_text`). Estas etiquetas aún están en formato de texto ('spam'/'ham').
    -   Lo mismo se hace para el conjunto de prueba (`X_test`  y  `y_test_text`).
3.  **Codificar Etiquetas 'spam'/'ham' a 1/0:**
    
    -   `label_encoder = LabelEncoder()`: Crea una instancia de  `LabelEncoder`  de scikit-learn. Esta herramienta es perfecta para convertir etiquetas categóricas (como 'spam' y 'ham') en valores numéricos.
    -   `y_train = label_encoder.fit_transform(y_train_text)`: Ajusta el codificador (`fit`) a las etiquetas de entrenamiento (`y_train_text`) para aprender las categorías únicas y luego las transforma (`transform`) a números. Por ejemplo, 'ham' podría ser 0 y 'spam' 1 (o viceversa, depende del orden).
    -   `y_test = label_encoder.transform(y_test_text)`: Aplica la misma transformación a las etiquetas de prueba (`y_test_text`) utilizando el  `LabelEncoder`  que ya se ajustó con los datos de entrenamiento. Es crucial usar el mismo codificador para asegurar la consistencia.
4.  **Imprimir Información del Dataset:**
    
    -   Los comandos  `print(f"Dataset de Entrenamiento: {len(X_train)} muestras")`, etc., muestran un resumen del tamaño de los datasets y cómo se han codificado las etiquetas, lo cual es útil para verificar que todo se cargó y procesó correctamente.

### **🧐 Análisis de la Necesidad de `LabelEncoder`**

Es lógico pensar  que las etiquetas en el CSV original ya podrían ser numéricas (0 y 1). Sin embargo, hay un detalle clave que hace que la línea de código con `LabelEncoder` sea **necesaria** (o, al menos, una práctica de programación segura) incluso si los datos ya parecen ser 0 y 1.

Aquí está el desglose:

#### **1. El Tipo de Dato en el DataFrame**

Cuando se lee un archivo CSV, pandas interpreta el tipo de dato de cada columna. Aunque los valores sean `0` y `1`, la columna `'label'` puede ser importada como un tipo de dato numérico (entero o flotante), pero **también puede ser importada como un _string_ o un tipo _categórico_** si el CSV tiene una estructura mixta o si pandas lo infiere de esa manera.

-   Si `y_train_text` (el contenido de `train_df['label']`) son **cadenas de texto** como `"0"` y `"1"`, entonces **sí se necesita** `LabelEncoder` o una conversión similar, porque TensorFlow Keras espera que las etiquetas de salida (`y_train`) sean tensores numéricos puros (enteros o flotantes) para calcular la función de pérdida.
    
-   Si `y_train_text` son **enteros puros** (`int64`), `LabelEncoder` técnicamente no es estrictamente necesario, pero tampoco daña, ya que `fit_transform` sobre enteros no cambiará los valores si son 0 y 1.
    

#### **2. Estandarización y Buenas Prácticas**

En el contexto de la librería `sklearn` y los problemas de clasificación binaria, usar `LabelEncoder` asegura una **conversión estándar y explícita** de las etiquetas categóricas a numéricas.

-   **Evita la Ambigüedad:** Garantiza que las etiquetas, independientemente de cómo se hayan cargado desde el CSV (como `str` o `int`), terminen siendo **el tipo de dato numérico y la codificación exacta** que el modelo de Keras necesita.
    
-   **Asegura la Mapeo Correcto:** Lo más importante es que `LabelEncoder` define explícitamente qué clase se mapea a **0** y cuál a **1**. Aunque en este caso parezca obvio (`0` a `0` y `1` a `1`), si el CSV contuviera etiquetas de texto como `"ham"` y `"spam"`, el `LabelEncoder` manejaría el mapeo de forma consistente.
    

### **💡 Conclusión (Revisando la Salida)**

La salida de tu código fue:

-   `Etiquetas originales: [0 1]`
    
-   `Codificación: [0 1]`
    

Esto **confirma** que la columna `'label'` fue interpretada por pandas como un conjunto de etiquetas categóricas, y que el `LabelEncoder` simplemente mapeó el valor original `0` al número `0` y el valor original `1` al número `1`.

**Respuesta Directa:**

Aunque los valores numéricos ya eran 0 y 1, el **uso de `LabelEncoder` es una medida de seguridad y estandarización** para asegurar que el tipo de dato de las etiquetas sea el adecuado para el entrenamiento de Keras, y que el mapeo de clases a números sea explícito y consistente entre el conjunto de entrenamiento (`fit_transform`) y el de prueba (`transform`).

### **3️⃣ Preprocesamiento del Texto: Vectorización**

Convertimos el texto en secuencias de números que el modelo puede procesar.

In [5]:
MAX_WORDS = 10000 # Número máximo de palabras del vocabulario
MAX_LEN   = 100   # Longitud máxima de las secuencias de texto

# 3.1 Definir la capa de vectorización
text_vectorization = TextVectorization(
    max_tokens=MAX_WORDS,
    output_mode='int',
    output_sequence_length=MAX_LEN
)

# 3.2 Adaptar la capa al texto de entrenamiento (construir el vocabulario)
text_vectorization.adapt(X_train)

# Mostrar las primeras palabras del vocabulario aprendido
print("\nPrimeras 100 palabras del vocabulario:")
print(text_vectorization.get_vocabulary()[:100])

# 3.3 Aplicar la vectorización a los conjuntos de datos
X_train_vectorized = text_vectorization(X_train)
X_test_vectorized = text_vectorization(X_test)

print("\nForma de X_train_vectorized:", X_train_vectorized.shape)
print("Primeras 5 secuencias vectorizadas de X_train:")
print(X_train_vectorized[:5])


Primeras 100 palabras del vocabulario:
['', '[UNK]', np.str_('de'), np.str_('que'), np.str_('a'), np.str_('en'), np.str_('no'), np.str_('y'), np.str_('la'), np.str_('el'), np.str_('un'), np.str_('es'), np.str_('para'), np.str_('por'), np.str_('lo'), np.str_('te'), np.str_('al'), np.str_('con'), np.str_('mi'), np.str_('me'), np.str_('una'), np.str_('tu'), np.str_('estoy'), np.str_('pero'), np.str_('ahora'), np.str_('las'), np.str_('si'), np.str_('se'), np.str_('o'), np.str_('más'), np.str_('mensaje'), np.str_('los'), np.str_('2'), np.str_('está'), np.str_('su'), np.str_('texto'), np.str_('bien'), np.str_('cuando'), np.str_('ltgt'), np.str_('del'), np.str_('4'), np.str_('qué'), np.str_('tarde'), np.str_('estás'), np.str_('casa'), np.str_('favor'), np.str_('así'), np.str_('como'), np.str_('sí'), np.str_('hola'), np.str_('solo'), np.str_('entonces'), np.str_('día'), np.str_('ya'), np.str_('semana'), np.str_('eso'), np.str_('tengo'), np.str_('porque'), np.str_('teléfono'), np.str_('este'),

Esta celda es fundamental para el  **preprocesamiento del texto**, donde convertimos el texto legible por humanos en un formato numérico que el modelo de Deep Learning pueda entender. Aquí te detallo cada parte:

1.  **`MAX_WORDS = 10000`  y  `MAX_LEN = 100`**:
    
    -   `MAX_WORDS`: Define el número máximo de palabras únicas que el modelo considerará para construir su vocabulario. Las palabras más frecuentes serán incluidas hasta este límite.
    -   `MAX_LEN`: Establece la longitud máxima de las secuencias de texto. Cada mensaje se truncará o se rellenará con ceros hasta alcanzar esta longitud. Esto asegura que todas las entradas al modelo tengan la misma dimensión.
2.  **`text_vectorization = TextVectorization(...)`  (3.1 Definir la capa de vectorización)**:
    
    -   Aquí creamos una instancia de la capa  `TextVectorization`  de TensorFlow. Esta capa es la encargada de estandarizar, tokenizar y vectorizar el texto.
    -   `max_tokens=MAX_WORDS`: Le decimos a la capa cuántas palabras únicas debe mantener en su vocabulario (las 10,000 más frecuentes).
    -   `output_mode='int'`: Indicamos que la salida de la capa deben ser secuencias de enteros, donde cada entero representa el índice de una palabra en el vocabulario.
    -   `output_sequence_length=MAX_LEN`: Aseguramos que todas las secuencias de salida tengan una longitud fija de 100, rellenando con ceros si son más cortas o truncando si son más largas.
3.  **`text_vectorization.adapt(X_train)`  (3.2 Adaptar la capa al texto de entrenamiento)**:
    
    -   Este es un paso crucial. La función  `adapt()`  analiza los datos de texto de  `X_train`  para:
        -   Construir el vocabulario, identificando las  `MAX_WORDS`  palabras más frecuentes.
        -   Calcular las estadísticas necesarias para la estandarización (por ejemplo, convertir a minúsculas, eliminar puntuación, etc.).
    -   Es importante adaptar la capa solo con los datos de entrenamiento para evitar la  _fuga de datos_  (data leakage) del conjunto de prueba.
4.  **`print(text_vectorization.get_vocabulary()[:100])`**:
    
    -   Muestra las primeras 100 palabras que la capa  `text_vectorization`  ha aprendido y añadido a su vocabulario, dándonos una idea de las palabras que el modelo 'reconoce'. El primer elemento es una cadena vacía y el segundo es  `[UNK]`  para palabras desconocidas.
5.  **`X_train_vectorized = text_vectorization(X_train)`  y  `X_test_vectorized = text_vectorization(X_test)`  (3.3 Aplicar la vectorización)**:
    
    -   Una vez que la capa  `text_vectorization`  ha sido adaptada, la usamos como una función para transformar los textos de entrenamiento (`X_train`) y de prueba (`X_test`) en sus respectivas representaciones numéricas (`X_train_vectorized`  y  `X_test_vectorized`).
6.  **`print("Forma de X_train_vectorized:", X_train_vectorized.shape)`  y  `print("Primeras 5 secuencias vectorizadas de X_train:")`**:
    
    -   Estas líneas imprimen la forma (dimensiones) del tensor resultante (`(número de muestras, MAX_LEN)`) y las primeras 5 secuencias de números, lo que permite verificar cómo se han transformado los textos.


## 1. 🔡 `TextVectorization` (Vectorización de Texto)

### **¿Qué es y Cuándo Ocurre?**

Es el **primer paso** en el _pipeline_ del procesamiento de lenguaje. Su trabajo es convertir las palabras humanas (cadenas de texto) en un formato numérico que el modelo de _Deep Learning_ pueda entender: **secuencias de números enteros**.

### **¿Cómo Funciona?**

1.  **Estandarización:** Limpia el texto (convierte todo a minúsculas, elimina signos de puntuación, etc.).
    
2.  **Construcción del Vocabulario (`.adapt(X_train)`):** Elige un número máximo de palabras frecuentes (`MAX_WORDS = 10000` en tu código) y les asigna un índice único.
    
    -   _Ejemplo:_
        
        -   La palabra "el" $\rightarrow$ índice 2
            
        -   La palabra "dinero" $\rightarrow$ índice 530
            
        -   La palabra "gratis" $\rightarrow$ índice 12
            
3.  **Mapeo del Texto:** Reemplaza cada palabra en el mensaje por su índice numérico correspondiente.
    

### **Salida de `TextVectorization`**

Una secuencia de números enteros de longitud fija.

> **Mensaje Original:** "Gana dinero facil y rapido desde casa."
>
> **Salida Vectorizada:** $[30, 530, 150, 4, 200, 45, 80, 0, 0, \dots]$
>
> _(Los ceros al final rellenan la secuencia hasta la longitud máxima `MAX_LEN = 100`.)_


### **4️⃣ Definición y Compilación del Modelo**

Creamos la arquitectura de la Red Neuronal, incluyendo la capa `Embedding` para aprender la representación vectorial de las palabras.

In [6]:
embedding_dim = 128 # Dimensión del vector de embeddings

model = models.Sequential([
    # La capa de Embedding toma la entrada de enteros de la vectorización
    Embedding(MAX_WORDS, embedding_dim),
    # Reduce la dimensionalidad promediando todos los embeddings de la secuencia
    GlobalAveragePooling1D(),
    # Primera Capa Oculta
    Dense(128, activation='relu'), # Mayor número de neuronas para aprender más
    Dropout(0.5),
    # Segunda Capa Oculta (Opcional, se puede quitar si solo quiere una)
    # Dense(64, activation='relu'),
    # Dropout(0.5), # Mantiene la regularización    # Capa de salida: 1 neurona con activación 'sigmoid' para clasificación binaria (0 o 1)
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # Función de pérdida para clasificación binaria
    metrics=['accuracy']
)

print("\n--- Resumen del Modelo ---")
model.summary()


--- Resumen del Modelo ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Esta celda define y compila el modelo de red neuronal que se utilizará para la clasificación de SPAM. Aquí te explico cada parte:

1.  **`embedding_dim = 128`**: Define la dimensión del vector de  _embeddings_  (incrustaciones) para cada palabra. Esto significa que cada palabra del vocabulario se representará como un vector de 128 números, que el modelo aprenderá para capturar el significado contextual.
    
2.  **`model = models.Sequential([...])`**: Esto crea un modelo de Keras que es una pila lineal de capas. Cada elemento dentro de la lista es una capa de la red neuronal:
    
    -   **`Embedding(MAX_WORDS, embedding_dim)`**: Esta es la primera capa. Toma las secuencias de enteros generadas por  `TextVectorization`  (donde cada entero representa una palabra).  `MAX_WORDS`  es el tamaño de nuestro vocabulario y  `embedding_dim`  es la dimensión que aprenderá para cada palabra. Convierte cada ID de palabra en su vector de  _embedding_  correspondiente.
    -   **`GlobalAveragePooling1D()`**: Después de la capa  `Embedding`, tenemos una secuencia de vectores (uno por palabra en la frase). Esta capa toma el promedio de todos los vectores de  _embedding_  en la secuencia, lo que resulta en un único vector de tamaño  `embedding_dim`  por cada mensaje. Esto reduce la dimensionalidad y permite pasar la información a capas densas.
    -   **`Dense(64, activation='relu')`**: Esta es una capa completamente conectada (_dense_) con 64 neuronas. La función de activación  `relu`  (Rectified Linear Unit) introduce no linealidad en el modelo, permitiéndole aprender patrones más complejos.
    -   **`Dropout(0.5)`**: Esta capa es una técnica de regularización. Durante el entrenamiento,  `Dropout`  apaga aleatoriamente el 50% de las neuronas de la capa anterior. Esto ayuda a prevenir el sobreajuste al hacer que el modelo sea menos dependiente de neuronas específicas.
    -   **`Dense(1, activation='sigmoid')`**: Esta es la capa de salida. Tiene 1 neurona porque estamos realizando una clasificación binaria (SPAM o NO SPAM). La función de activación  `sigmoid`  comprime la salida a un valor entre 0 y 1, que puede interpretarse como la probabilidad de que un mensaje sea SPAM.
3.  **`model.compile(...)`**: Este paso configura el modelo para el entrenamiento. Aquí se especifican:
    
    -   **`optimizer='adam'`**: El algoritmo de optimización que ajustará los pesos del modelo durante el entrenamiento para minimizar la función de pérdida. 'Adam' es un optimizador muy popular y eficaz.
    -   **`loss='binary_crossentropy'`**: La función de pérdida que el modelo intentará minimizar.  `binary_crossentropy`  es la función de pérdida estándar para problemas de clasificación binaria.
    -   **`metrics=['accuracy']`**: Las métricas que se utilizarán para evaluar el rendimiento del modelo durante el entrenamiento y la evaluación. En este caso, la 'accuracy' (precisión) mide la proporción de predicciones correctas.
4.  **`model.summary()`**: Imprime un resumen de la arquitectura del modelo, mostrando las capas, la forma de salida de cada capa y el número de parámetros entrenables.

## 2.  `Embedding` (Incrustación de Palabras)

### **¿Qué es y Cuándo Ocurre?**

Es la **primera capa de la red neuronal** que toma la secuencia de índices enteros generada por `TextVectorization`. Su función es convertir cada índice (que por sí solo no tiene significado) en un **vector de números reales** llamado _embedding_.

### **¿Cómo Funciona?**

1.  **Representación Densa:** En lugar de usar un solo número entero (el índice), el _embedding_ utiliza un vector de números flotantes (con una dimensión de, por ejemplo, `embedding_dim = 128` en tu código).
    
2.  **Aprendizaje Semántico:** Durante el entrenamiento, el modelo ajusta los valores dentro de este vector para que las palabras con significados similares (ej. "dinero" y "premio") tengan vectores _parecidos_ en el espacio de 128 dimensiones.
    

### **Salida de `Embedding`**

Una **matriz tridimensional** que representa todo el mensaje:

-   **Dimensión 1 (Batch):** El número de mensajes que se procesan juntos (ej. 32).
    
-   **Dimensión 2 (Longitud de la secuencia):** El número de palabras en el mensaje (`MAX_LEN = 100`).
    
-   **Dimensión 3 (Dimensionalidad del Embedding):** El tamaño del vector que representa cada palabra (`embedding_dim = 128`).
    

> **Matriz de Salida (por cada mensaje):** $\text{Tamaño} = (\text{Longitud de la Secuencia}, \text{Dimensión del Embedding}) = (100, 128)$

----------

## 3. 📉 `GlobalAveragePooling1D` (Reducción de Dimensión)

### **¿Qué es y Cuándo Ocurre?**

Es la capa que viene **inmediatamente después del _Embedding_**. Su función es reducir la complejidad de la matriz 3D generada por el _Embedding_ a un simple vector 1D de tamaño fijo. Esto es crucial porque la siguiente capa (`Dense`) solo puede aceptar entradas de vector fijo.

### **¿Cómo Funciona?**

Realiza un **promedio simple** de todos los vectores de _embedding_ a lo largo de la dimensión de la secuencia (la dimensión de 100). Es decir:

1.  Toma el mensaje (que es una matriz de $100 \times 128$).
    
2.  Para la primera dimensión del _embedding_ (la posición 0 de los 128), promedia los 100 valores.
    
3.  Repite esto para las 128 dimensiones.
    

Esto reduce la información de todas las palabras a un solo vector que resume el "sentido general" del mensaje.

### **Salida de `GlobalAveragePooling1D`**

Un **vector bidimensional** (una lista de números) de tamaño fijo:

-   **Dimensión 1 (Batch):** El número de mensajes.
    
-   **Dimensión 2 (Características):** El tamaño del _embedding_ (`128`).
    

> **Vector de Salida (por cada mensaje):** $\text{Tamaño} = (\text{Dimensión del Embedding}) = (128)$

Este vector de 128 características es el que finalmente se alimenta a la capa `Dense(64, activation='relu')` para que el modelo tome la decisión final de **SPAM** o **HAM**.

### **Resumen del Flujo de Datos**

| Capa | Entrada | Salida | Función |
| :---- | :---- | :---- | :---- |
| **TextVectorization** | Texto crudo (str) | Secuencia de índices (int) | Convierte palabras en números. |
| **Embedding** | Secuencia de índices (int) | Secuencia de vectores (Matriz $100 \\times 128$) | Convierte números en vectores de significado semántico. |
| **GlobalAveragePooling1D** | Secuencia de vectores (Matriz $100 \\times 128$) | Vector de características (Vector $1 \\times 128$) | Resume todo el mensaje en un vector fijo. |



## **🔬 Ejemplo Mínimo: Procesamiento de una Frase**

### **1\. 🔡 Capa: TextVectorization**

**Configuración Simplificada:**

* **Vocabulario aprendido:** {"\[UNK\]": 1, "el": 2, "premio": 3, "ganar": 4}  
  * *Nota:* 0 se usa para *padding* (relleno) y 1 para *Out-Of-Vocabulary* (palabras no aprendidas, \[UNK\]).  
* **Longitud Máxima de Secuencia (MAX\_LEN):** 5

#### **A. Entrada y Mapeo**

| Frase de Entrada | "quiero" | "ganar" | "el" | "premio" |
| :---- | :---- | :---- | :---- | :---- |

#### **B. Salida de TextVectorization (Índices)**

El mensaje se convierte en una secuencia de índices.

| Índice | 1 | 4 | 2 | 3 | 0 |
| :---- | :---- | :---- | :---- | :---- | :---- |
| **Palabra** | \[UNK\] | ganar | el | premio | \[PAD\] |

*   
  *"Quiero"* no está en el vocabulario, se mapea a \[UNK\] (1).  
* Se añade un \[PAD\] (0) al final para completar la longitud de 5 (MAX\_LEN).

---

### **2\. Capa: Embedding**

El objetivo ahora es convertir cada índice (1, 4, 2, 3, 0) en un vector de números reales.

**Configuración Simplificada:**

* **Dimensión del *Embedding* (embedding\_dim):** 3 (En lugar de 128, para que quepa en la pantalla).  
* El *Embedding* es una tabla de consulta. Asumamos que, tras el entrenamiento, los vectores de pesos aprendidos son:

| Índice (Palabra) | Vector de 3 Dimensiones (Características) |
| :---- | :---- |
| **0 (\[PAD\])** | \[0.0, 0.0, 0.0\] |
| **1 (\[UNK\])** | \[0.1, \-0.2, 0.5\] |
| **2 (el)** | \[0.6, 0.7, \-0.3\] |
| **3 (premio)** | \[0.9, 0.8, \-0.1\] |
| **4 (ganar)** | \[0.5, 0.4, 0.0\] |



#### **Salida de Embedding (Matriz de Vectores)**

La secuencia de índices se transforma en una matriz de $5 * 3$.


$$\begin{array}{c|ccc}
\textbf{Posición} & \textbf{Dim 1} & \textbf{Dim 2} & \textbf{Dim 3} \\
\hline
\text{1 (quiero/[UNK])} & 0.1 & -0.2 & 0.5 \\
\text{2 (ganar)} & 0.5 & 0.4 & 0.0 \\
\text{3 (el)} & 0.6 & 0.7 & -0.3 \\
\text{4 (premio)} & 0.9 & 0.8 & -0.1 \\
\text{5 ([PAD])} & 0.0 & 0.0 & 0.0 \\
\end{array}$$

---

### **3\. 📉 Capa: GlobalAveragePooling1D**

El objetivo es reducir la matriz de $5 * 3$ a un único vector de 3 dimensiones, promediando la información de todas las palabras.


#### **A. Cálculo del Promedio**

Simplemente calculamos el promedio de todos los valores en **cada columna** (dimensión) de la matriz anterior.

-   Promedio Dimensión 1:
    
    $$(0.1 + 0.5 + 0.6 + 0.9 + 0.0) / 5 = 2.1 / 5 = 0.42$$
    
-   Promedio Dimensión 2:
    
    $$(-0.2 + 0.4 + 0.7 + 0.8 + 0.0) / 5 = 1.7 / 5 = 0.34$$
    
-   Promedio Dimensión 3:
    
    $$(0.5 + 0.0 + (-0.3) + (-0.1) + 0.0) / 5 = 0.1 / 5 = 0.02$$
    

#### **B. Salida Final de `GlobalAveragePooling1D` (Vector Resumen)**

El mensaje completo se ha condensado en un solo vector de 3 características:

$$\mathbf{[0.42, 0.34, 0.02]}$$

#### **C. ¿Qué Significa este Vector?**

Este vector de 3 números es el **resumen numérico de todo el mensaje**. Este vector de 3 características (que en tu modelo real es de 128 características) es lo que se alimenta a la primera capa **`Dense(64, activation='relu')`** para que el modelo decida si la combinación de estos números (la semántica del mensaje) es más parecida al SPAM o al HAM.

### **Pero una palabra es única. Cual es la nesecidad o ventaja de representarla por 3 o 128 caracteristicas en lugar de una?**
____
Es la pregunta central de por qué el **Deep Learning** es tan poderoso en el Procesamiento de Lenguaje Natural (PLN). Tienes toda la razón: una palabra es única y el paso de `TextVectorization` le asigna un índice único (un solo número).

Sin embargo, usar un solo número (el índice) tiene serios inconvenientes. La capa `Embedding` soluciona esto al representarla con 3, 128 (o incluso más) características.

Aquí te explico la **necesidad y la enorme ventaja** de esta representación vectorial (el _Embedding_):

----------

## 🔑 La Necesidad de Múltiples Características (Vectorización Densa)

### 1. Problema: El Índice es una Ficha de Identidad, No una Descripción

Cuando a una palabra le asignas un solo índice:
| Palabra | Índice (Ficha ID) |
| :---- | ----- |
| dinero | 530 |
| premio | 531 |
| gato | 532 |

-   **No hay Relación Semántica:** Para el modelo, el número `530` está **tan cerca** del `531` como lo está del `532`. El modelo interpreta que la diferencia entre "dinero" (530) y "premio" (531) es la misma que entre "dinero" (530) y "gato" (532). Esto es incorrecto, ya que **dinero y premio están semánticamente relacionados** (ambos se asocian con SPAM, ganancias, etc.), mientras que "gato" no lo está.
    
-   **Dimensionalidad Blanda:** Usar el índice único es una forma de codificación dispersa (_sparse encoding_), donde la única información que tiene el modelo es si la palabra _existe_ o _no existe_. No le da herramientas para entender el significado.
    

----------

## 🌟 La Ventaja del _Embedding_ (Representación Densa)

Al representar la palabra con un vector de 128 números (características), transformamos el problema de una simple identificación a un **problema de ubicación en el espacio**.

### 1. Captura del Significado Semántico (Relaciones)

Cada una de las 128 características del vector se aprende durante el entrenamiento para codificar un **aspecto del significado** de la palabra.

-   **Palabras Relacionadas están Cercanas:** Si tomas el vector de 128 dimensiones para "dinero" y el vector para "premio", verás que, en el espacio de 128 dimensiones, estos dos vectores están **muy juntos**. En cambio, el vector para "gato" estará muy lejos.
    
-   Permite Operaciones Vectoriales: El modelo puede entender relaciones complejas. Se ha demostrado que en un embedding bien entrenado, se cumplen relaciones análogas como:
    
    $$\vec{\text{rey}} - \vec{\text{hombre}} + \vec{\text{mujer}} \approx \vec{\text{reina}}$$
    

### 2. Mayor Capacidad de Predicción

Para la tarea de **Detección de SPAM**, estas 128 características permiten al modelo:

-   **Generalizar:** Si el modelo nunca vio la palabra "ganancia", pero vio la palabra "beneficio", y ambas tienen vectores muy cercanos, el modelo puede clasificar un mensaje con "ganancia" correctamente como SPAM.
    
-   **Distinguir Contextos:** Las diferentes dimensiones del vector pueden aprender qué tan positiva o negativa es una palabra, qué tan formal o informal es, o si está relacionada con transacciones financieras o animales domésticos.
    

### 3. El Beneficio de `GlobalAveragePooling1D`

Cuando aplicas `GlobalAveragePooling1D`, estás promediando **todos los vectores de 128 características** del mensaje.

-   **Resultado:** Obtienes un **solo vector de 128 características que resume la semántica promedio de todo el mensaje**.
    
-   **Ejemplo:** Si tu mensaje incluye muchas palabras de SPAM (premio, clic, urgente), el vector promedio se moverá hacia la región del espacio de 128D donde residen los mensajes de SPAM, facilitando a la capa `Dense` la clasificación.



| Característica | Representación por Índice Único (Mala) | Representación por Vector Densa (Embedding) (Buena) |
| :---- | :---- | :---- |
| **Significado** | Solo dice quién es la palabra. | Dice qué significa la palabra y con qué se relaciona. |
| **Relación Numérica** | Distancias incorrectas (530 vs 531). | Distancias correctas (dinero y premio están muy cerca). |
| **Capacidad del Modelo** | Muy limitada. | Alta; puede generalizar y entender el contexto. |


### **5️⃣ Entrenamiento del Modelo**

Entrenamos el modelo con los datos vectorizados.

In [7]:
# Número de pasadas completas sobre el conjunto de entrenamiento
EPOCHS = 50
# Número de muestras procesadas antes de actualizar los pesos
BATCH_SIZE = 32

print("\n--- Iniciando Entrenamiento ---")
history = model.fit(
    X_train_vectorized,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_test_vectorized, y_test)
)


--- Iniciando Entrenamiento ---
Epoch 1/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.8620 - loss: 0.3708 - val_accuracy: 0.8664 - val_loss: 0.3425
Epoch 2/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.8726 - loss: 0.3011 - val_accuracy: 0.9238 - val_loss: 0.1998
Epoch 3/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9547 - loss: 0.1369 - val_accuracy: 0.9623 - val_loss: 0.1049
Epoch 4/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9749 - loss: 0.0789 - val_accuracy: 0.9785 - val_loss: 0.0747
Epoch 5/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9836 - loss: 0.0589 - val_accuracy: 0.9794 - val_loss: 0.0708
Epoch 6/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9868 - loss: 0.0474 - val_accuracy: 0.9740 - val_loss: 0.0745
Epoch 7/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9870 - loss: 0.0442 - val_accuracy: 0.9408 - val_loss: 0.1883
Epoch 8/50
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy:

Esta celda se encarga de entrenar el modelo de red neuronal que definimos previamente. Aquí te explico los detalles:

-   **`EPOCHS = 100`**: Define el número de  `epochs`  (épocas). Una época significa que el modelo procesa todo el conjunto de entrenamiento una vez. En cada época, el modelo ajusta sus pesos para intentar mejorar la predicción. Un número alto de épocas puede llevar a un buen aprendizaje, pero también al sobreajuste si no se maneja correctamente.
-   **`BATCH_SIZE = 32`**: Define el tamaño del  `batch`  (lote). Durante el entrenamiento, los datos no se alimentan al modelo de uno en uno, sino en grupos más pequeños llamados lotes. Un  `batch_size`  de 32 significa que el modelo procesará 32 muestras a la vez antes de actualizar sus pesos. Esto es un compromiso entre la velocidad de entrenamiento y la estabilidad del descenso de gradiente.
-   **`print("--- Iniciando Entrenamiento ---")`**: Simplemente un mensaje para indicar que el proceso de entrenamiento ha comenzado.
-   **`history = model.fit(...)`**: Esta es la función principal que inicia el entrenamiento del modelo. Recibe los siguientes parámetros:
    -   **`X_train_vectorized`**: Son los datos de entrenamiento (los mensajes de texto vectorizados).
    -   **`y_train`**: Son las etiquetas de verdad (0 para 'ham', 1 para 'spam') correspondientes a  `X_train_vectorized`.
    -   **`epochs=EPOCHS`**: Usa el número de épocas que definimos (100).
    -   **`batch_size=BATCH_SIZE`**: Usa el tamaño de lote que definimos (32).
    -   **`validation_data=(X_test_vectorized, y_test)`**: Aquí le indicamos al modelo que, al final de cada época, debe evaluar su rendimiento en el conjunto de datos de validación (`X_test_vectorized`  y  `y_test`). Esto es crucial para monitorear si el modelo está sobreajustando los datos de entrenamiento (es decir, si su rendimiento mejora en el entrenamiento pero empeora o se estanca en la validación).

### **6️⃣ Prueba del Modelo**

Definimos una función para vectorizar texto nuevo y realizar predicciones.

In [8]:
def predict_spam(text):
    # Vectorizar el texto de entrada (debe ser una lista o tensor, por eso [text])
    vectorized_text = text_vectorization([text])

    # Realizar la predicción (devuelve un array, tomamos el primer elemento [0][0])
    prediction = model.predict(vectorized_text, verbose=0) # verbose=0 para no mostrar el log
    probability = prediction[0][0]

    # Clasificación final
    if probability > 0.5:
        label = "SPAM"
    else:
        label = "NO SPAM (HAM)"

    return label, probability

# Ejemplos de prueba
example_texts = [
    "¡Felicidades! Has ganado un premio de 1.000.000 euros. Haz clic aquí para reclamar.",
    "Hola, ¿estamos libres para tomar un café esta tarde?",
    "Gana dinero facil y rapido desde casa. Contacta ahora mismo.",
    "¿Puedes pasarme la dirección de correo electrónico de Ana, por favor?",
    "Mensaje urgente: Su cuenta ha sido suspendida. Verifique su identidad en este enlace."
]

print("\n--- Resultados de la Predicción ---")
for text in example_texts:
    label, probability = predict_spam(text)
    print(f"Texto: '{text}'")
    print(f"Predicción: **{label}** (Probabilidad: {probability:.4f})\n")


--- Resultados de la Predicción ---
Texto: '¡Felicidades! Has ganado un premio de 1.000.000 euros. Haz clic aquí para reclamar.'
Predicción: **SPAM** (Probabilidad: 0.9978)

Texto: 'Hola, ¿estamos libres para tomar un café esta tarde?'
Predicción: **NO SPAM (HAM)** (Probabilidad: 0.0000)

Texto: 'Gana dinero facil y rapido desde casa. Contacta ahora mismo.'
Predicción: **SPAM** (Probabilidad: 0.5737)

Texto: '¿Puedes pasarme la dirección de correo electrónico de Ana, por favor?'
Predicción: **NO SPAM (HAM)** (Probabilidad: 0.0000)

Texto: 'Mensaje urgente: Su cuenta ha sido suspendida. Verifique su identidad en este enlace.'
Predicción: **SPAM** (Probabilidad: 0.9998)



<HASTA AQUI>

Esta celda de código se encarga de probar el modelo entrenado con nuevos mensajes de texto.
Primero, define una función llamada `predict_spam` que toma un mensaje de texto como entrada. Dentro de esta función, el texto se vectoriza utilizando la capa `text_vectorization` que se adaptó previamente a los datos de entrenamiento. Luego, el modelo (`model`) utiliza esta representación numérica del texto para hacer una predicción de probabilidad de SPAM.
Si esta probabilidad es mayor a 0.5, el mensaje se clasifica como 'SPAM'; de lo contrario, como 'NO SPAM (HAM)'.
Finalmente, la celda itera a través de una lista de `example_texts` predefinidos, llama a la función `predict_spam` para cada uno y muestra el texto original, la clasificación (SPAM/NO SPAM) y la probabilidad calculada por el modelo.

In [9]:
# Crear un modelo que incluya la vectorización
export_model = tf.keras.Sequential([
  text_vectorization,
  model
])

# Compilarlo (opcional para inferencia, pero recomendado)
export_model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

# Guardar el modelo en un archivo
export_model.save('modelo_spam_completo.keras')

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [10]:
!python --version

Python 3.12.12
